## 3. Pré-Processamento (Preparação para Vetoriazação)

- **Input**: CSV contendo todos os enunciados, alternativas e gabaritos

- **Output**: CSV com as colunas pré-processadas -> _numero_questao_, _enunciado_, _alternativas_,_nu_param_B_, _gabarito_, _ano_, _enunciado_limpo_ e _alternativas_limpo_


In [27]:
# Importando dependências para o pré-processamento dos dados
import pandas as pd
import numpy as np
import re
import random
import nltk
from nltk.corpus import stopwords

# Baixando stopwords do NLTK
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/renan_gonzales/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [28]:
# Lendo os dados cruzados do ENEM
enem_df = pd.read_csv("../data/final/enem_data.csv")
enem_df.head()

,numero_questao,enunciado,alternativas,nu_param_B,gabarito,questao,pc_amostra_acertos,ano
0,1,A atmosfera terrestre é composta pelos gases n...,A: reduzir o calor irradiado pela Terra median...,-1.70677,C,1,0.93,2009
1,2,Analise a figura. Supondo que seja necessário ...,A: Concentração média de álcool no sangue ao l...,0.62043,D,2,0.40,2009
2,3,Estima-se que haja atualmente no mundo 40 milh...,"A: induzir a imunidade, para proteger o organi...",2.07704,A,3,0.40,2009
3,4,"Em um experimento, preparou-se um conjunto de ...",A: os genótipos e os fenótipos idênticos.; B: ...,0.11500,B,4,0.61,2009
4,5,"Na linha de uma tradição antiga, o astrônomo g...",A: Ptolomeu apresentou as ideias mais valiosas...,0.21694,E,5,0.60,2009


## 3.1. Removendo Questões com Alternativas Vazias (Figuras)


In [29]:
def remove_empty_alternatives(alternativas):
    parts = [p.strip() for p in alternativas.split(";") if p.strip()]
    cleaned = []

    for part in parts:
        if ":" not in part:
            continue
        label, text = part.split(":", 1)
        if text.strip():
            cleaned.append(f"{label.strip()}: {text.strip()}")
    return "; ".join(cleaned)


enem_df["alternativas"] = enem_df["alternativas"].apply(remove_empty_alternatives)
enem_df = enem_df[enem_df["alternativas"].astype(bool)].reset_index(drop=True)

## 3.2. Extraindo Gabarito e Distratores


In [30]:
def separate_distractors(row):
    items: list = [
        alternativa.strip()
        for alternativa in row["alternativas"].split(";")
        if alternativa.strip()
    ]

    mapping_items = {}
    for item in items:
        letra, texto = item.split(":", 1)
        mapping_items[letra.strip()] = texto.strip()

    # Extraindo texto do gabarito
    gabarito: str = row["gabarito"]
    row["gabarito_texto"] = mapping_items.get(gabarito.strip(), "")

    # Coletando alternativas distratoras
    row["distratores"] = "; ".join(
        [texto for letra, texto in mapping_items.items() if letra != gabarito]
    )

    return row


enem_df = enem_df.apply(separate_distractors, axis=1)

## 3.3. Aplicação do Protocolo Primi (2021)


In [31]:
# Coletando stopwords em português
pt_stopwords = set(stopwords.words("portuguese"))

random.sample(sorted(pt_stopwords), 10)

['há',
 'tivesse',
 'te',
 'houverão',
 'tivéssemos',
 'fui',
 'formos',
 'seriam',
 'tiver',
 'tenha']

In [32]:
def apply_protocolo_primi(text: str) -> list[str]:
    """
    Processando texto de acordo com o Protocolo Primi (2021), citado no artigo:
    Dado um texto, divide em palavras, transforma em minúsculas, remove números, stopwords e duplicatas.
    """
    # 1. Divide em palavras transformando em minúsculas
    tokens = re.findall(r"\b\w+\b", text.lower())

    # 2. Filtrando stopwords e palavras não alfabéticas
    filtered = [
        token for token in tokens if token.isalpha() and token not in pt_stopwords
    ]

    # 3. Removendo duplicatas
    return " ".join(filtered)

In [33]:
# Aplicando protocolo de limpeza nos dados
enem_df["enunciado_tokens"] = enem_df["enunciado"].apply(apply_protocolo_primi)
enem_df["gabarito_tokens"] = enem_df["gabarito_texto"].apply(apply_protocolo_primi)
enem_df["distratores_tokens"] = enem_df["distratores"].apply(apply_protocolo_primi)

## 3.4. Removendo Caracteres Especiais


In [34]:
caracteres_a_remover = {",", ".", ":", ";", "(", ")"}

for char in caracteres_a_remover:
    enem_df["gabarito_tokens"] = enem_df["gabarito_tokens"].str.replace(
        char, "", regex=False
    )
    enem_df["enunciado_tokens"] = enem_df["enunciado_tokens"].str.replace(
        char, "", regex=False
    )
    enem_df["distratores_tokens"] = enem_df["distratores_tokens"].str.replace(
        char, "", regex=False
    )

In [35]:
# Removendo linhas que estão vazias após a remoção de caracteres especiais
enem_df.dropna(
    subset=["gabarito_tokens", "enunciado_tokens", "distratores_tokens"], inplace=True
)
enem_df = enem_df[
    enem_df[["gabarito_tokens", "enunciado_tokens", "distratores_tokens"]]
    .apply(lambda x: all(x.str.strip() != ""), axis=1)
]

## 3.5. Verificando Dados Ausentes


In [36]:
enem_df.isna().sum()

numero_questao        0
enunciado             0
alternativas          0
nu_param_B            0
gabarito              0
questao               0
pc_amostra_acertos    0
ano                   0
gabarito_texto        0
distratores           0
enunciado_tokens      0
gabarito_tokens       0
distratores_tokens    0
dtype: int64

## 3.6. Salvando Dados Pré-Processados


In [37]:
# Adicionando a coluna de dificuldade ao DataFrame
dificuldade = enem_df.pop("nu_param_B")
enem_df["dificuldade"] = dificuldade

enem_df.to_csv("../data/final/cleaned_enem_data.csv", index=False)
print("Processing complete. File saved as 'cleaned_enem_data.csv'.")

Processing complete. File saved as 'cleaned_enem_data.csv'.


---